In [ ]:
import math
import os
import glob

import onnx
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

import onnx2tf
import tensorflow as tf
print("TensorFlow version: ", tf.__version__)

In [ ]:
def load_model(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter

In [ ]:
def model2onnx(file_path: str, model: nn.Module, x: np.ndarray, simplify=False):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    torch.onnx.export(
        model,
        torch.Tensor(x),
        file_path,
        input_names=["input_0"],
        output_names=["output_0"],
    )

    if simplify:
        import onnxslim

        model_onnx = onnx.load(file_path)
        model_onnx = onnxslim.slim(model_onnx)
        onnx.save(model_onnx, file_path)

In [ ]:
class Dummy(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigm = nn.Sigmoid()

    def forward(self, x):
        return self.sigm(x)

In [ ]:
LIMIT_MIN = -10
LIMIT_MAX = 2
x = torch.linspace(LIMIT_MIN, LIMIT_MAX, 100, dtype=torch.float32)
print(x.shape)

model = Dummy()
model.eval()
y = model(x[None])

plt.plot(x.numpy(), y.detach().numpy()[0])

In [ ]:
model2onnx("dummy_saved_model/dummy.onnx", model, x.numpy()[None], simplify=True)

In [ ]:
# x.numpy()[None]
# x_norm = (x - x.min()) / (x.max() - x.min())

# np.save("dummy_saved_model/x_norm.npy", x_norm.numpy())
# np.save("dummy_saved_model/x.npy", x.numpy())

# onnx2tf.convert(
#     input_onnx_file_path="dummy_saved_model/dummy.onnx",
#     output_folder_path="dummy_saved_model",
#     not_use_onnxsim=True,
#     verbosity=True,
#     # keep_ncw_or_nchw_or_ncdhw_input_names=["input_0"],
#     output_integer_quantized_tflite=True,
#     input_output_quant_dtype="uint8",
#     quant_type="per-tensor",
#     custom_input_op_name_np_data_path=[[
#         "input_0", "dummy_saved_model/x_norm.npy", [[0]], [[1]]
#     ]],
#     disable_group_convolution=True,  # for end-to-end model compatibility
#     enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
# )

In [ ]:
# x.numpy()[None]
# x_norm = (x - x.min()) / (x.max() - x.min()) * 255

# np.save("dummy_saved_model/x_norm.npy", x_norm.numpy()[None])
# np.save("dummy_saved_model/x.npy", x.numpy())

# onnx2tf.convert(
#     input_onnx_file_path="dummy_saved_model/dummy.onnx",
#     output_folder_path="dummy_saved_model",
#     not_use_onnxsim=True,
#     verbosity=True,
#     copy_onnx_input_output_names_to_tflite=True,
#     output_integer_quantized_tflite=True,
#     input_output_quant_dtype="uint8",
#     quant_type="per-tensor",
#     custom_input_op_name_np_data_path=[[
#         "input_0", "dummy_saved_model/x_norm.npy", [[127.5]], [[25]]
#     ]],
#     disable_group_convolution=True,  # for end-to-end model compatibility
#     enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
# )

In [ ]:
# # x_norm = (x - x.min()) / (x.max() - x.min()) * 255
# x_min = 0
# x_max = 1
# x_norm = (x - x_min) / (x_max - x_min) * 255

# np.save("dummy_saved_model/x_norm.npy", x_norm.numpy()[None])
# np.save("dummy_saved_model/x.npy", x.numpy()[None])

# onnx2tf.convert(
#     input_onnx_file_path="dummy_saved_model/dummy.onnx",
#     output_folder_path="dummy_saved_model",
#     not_use_onnxsim=True,
#     verbosity=True,
#     copy_onnx_input_output_names_to_tflite=True,
#     output_integer_quantized_tflite=True,
#     input_output_quant_dtype="uint8",
#     quant_type="per-tensor",
#     custom_input_op_name_np_data_path=[[
#         "input_0", "dummy_saved_model/x_norm.npy", [[0]], [[255]]
#     ]],
#     disable_group_convolution=True,  # for end-to-end model compatibility
#     enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
# )

In [ ]:
# a = np.random.randn(100)

# a = (a - a.min()) / (a.max() - a.min()) * 255

# np.save("dummy_saved_model/a.npy", a[None])

# onnx2tf.convert(
#     input_onnx_file_path="dummy_saved_model/dummy.onnx",
#     output_folder_path="dummy_saved_model",
#     not_use_onnxsim=True,
#     verbosity=True,
#     copy_onnx_input_output_names_to_tflite=True,
#     output_integer_quantized_tflite=True,
#     input_output_quant_dtype="uint8",
#     quant_type="per-tensor",
#     custom_input_op_name_np_data_path=[[
#         "input_0", "dummy_saved_model/a.npy", [[0]], [[255]]
#     ]],
#     disable_group_convolution=True,  # for end-to-end model compatibility
#     enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
# )

In [ ]:
x_norm = (x - x.min()) / (x.max() - x.min()) * 255


np.save("dummy_saved_model/x_norm.npy", x_norm.numpy()[None])
np.save("dummy_saved_model/x.npy", x.numpy()[None])

onnx2tf.convert(
    input_onnx_file_path="dummy_saved_model/dummy.onnx",
    output_folder_path="dummy_saved_model",
    not_use_onnxsim=True,
    verbosity=True,
    copy_onnx_input_output_names_to_tflite=True,
    output_integer_quantized_tflite=True,
    input_output_quant_dtype="uint8",
    quant_type="per-tensor",
    custom_input_op_name_np_data_path=[[
        "input_0", "dummy_saved_model/x_norm.npy", [[255 * np.abs(LIMIT_MIN) / (LIMIT_MAX - LIMIT_MIN)]], [[255 / (LIMIT_MAX - LIMIT_MIN)]]  # 255 / 2 for mean, 255 / (LIMIT * 2) for std
    ]],
    disable_group_convolution=True,  # for end-to-end model compatibility
    enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
)

In [ ]:
x_norm.mean()

In [ ]:
model_fp32 = load_model("dummy_saved_model/dummy_float32.tflite")
model_uint8 = load_model("dummy_saved_model/dummy_full_integer_quant.tflite")

In [ ]:
def infer(model, x, dequantize=True):
    input_details = model.get_input_details()[0]
    output_details = model.get_output_details()[0]

    model.set_tensor(input_details['index'], x)
    model.invoke()
    outputs = model.get_tensor(output_details['index'])

    scale, zero_point = output_details['quantization']
    if scale != 0.0 and dequantize:
        print(f"Dequantizing with scale={scale}, zero_point={zero_point}")
        outputs = (outputs.astype(np.float32) - zero_point) * scale  # re-scale

    return outputs

In [ ]:
y_fp32 = infer(model_fp32, x[None])[0]
# y_uint8 = infer(model_uint8, (x_norm[None].numpy()).astype(np.uint8))[0]
# y_uint8 = infer(model_uint8, (np.array(sorted(a))[None]).astype(np.uint8))[0]
y_uint8 = infer(model_uint8, (x_norm[None].numpy()).astype(np.uint8))[0]

plt.figure(figsize=(15, 5))
plt.plot(x.numpy(), y[0], label='PyTorch Sigmoid')
plt.plot(x.numpy(), y_fp32, '--', label='TFLite FP32 Sigmoid')
# plt.plot(np.array(sorted(a)) / 255, y_uint8, label='TFLite UINT8 Sigmoid')
plt.plot(x.numpy(), y_uint8, label='TFLite UINT8 Sigmoid')

plt.legend()

In [ ]:
input3 = np.load("head_uint8_saved_model/input_3.npy")
input4 = np.load("head_uint8_saved_model/input_4.npy")
input2 = np.load("head_uint8_saved_model/input_2.npy")

input_tensor = np.hstack([input3, input4, input2])
input_tensor.shape

In [ ]:
input_tensor.min(), input_tensor.max()

In [ ]:
input_tensors = []
for i in glob.glob("head_uint8_quant_input_norm_saved_model/input_*_quantized.npy"):
    input_tensors.append(np.load(i))
    
for i in input_tensors:
    print(i.shape, i.min(), i.max())
    
for i in input_tensors:
    plt.figure(figsize=(3, 2))
    plt.hist(i.flatten(), bins=100)
    plt.title(f"Histogram of {i.shape}")
    plt.xlabel("Value")
    plt.ylabel("Frequency")

In [ ]:
input_tensors = []
for i in range(7):
    input_tensors.append(np.load(f"head_uint8_saved_model/input_{i}.npy"))
    
for i in input_tensors:
    print(i.shape, i.min(), i.max())
    
for i in input_tensors:
    plt.figure(figsize=(3, 2))
    plt.hist(i.flatten(), bins=100)
    plt.title(f"Histogram of {i.shape}")
    plt.xlabel("Value")
    plt.ylabel("Frequency")